# 1 - TMDB extraction

Verified resume from 3000 cached movie details. No new API requests were needed. The request error-handling branches were not exercised against the live API.

In [19]:
from pathlib import Path
import os

project_root = Path.cwd()
if not (project_root / "cahier_des_charges.txt").exists():
    project_root = project_root.parent
if not (project_root / "cahier_des_charges.txt").exists():
    raise RuntimeError("Open this notebook from the project or notebooks folder.")
os.chdir(project_root)


## Imports and setup

In [20]:
import json
import os
import tempfile
import time

import requests
from dotenv import load_dotenv


TARGET_MOVIES = 3000
RAW_FILE = "data/raw/tmdb_raw.json"

load_dotenv()
token = os.getenv("TMDB_TOKEN")
if not token:
    raise SystemExit("Add TMDB_TOKEN to .env first.")

headers = {"Authorization": f"Bearer {token}"}

## Request data

In [21]:
def get_json(url, params):
    for attempt in range(1, 4):
        response = requests.get(url, headers=headers, params=params, timeout=20)
        if response.status_code == 429:
            print("Too many requests. Waiting 10 seconds. Attempt", attempt, "of 3")
            time.sleep(10)
            continue
        response.raise_for_status()
        data = response.json()
        if not data:
            raise ValueError("TMDB returned an empty response.")
        return data
    raise RuntimeError("TMDB is still limiting requests. Try again later.")

## Resume saved data

In [22]:
if os.path.exists(RAW_FILE):
    with open(RAW_FILE, encoding="utf-8") as file:
        raw = json.load(file)
else:
    raw = {"pages": {}, "movie_details": {}}

## Save progress

In [23]:
def save_raw():
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", delete=False) as file:
        json.dump(raw, file, ensure_ascii=False, indent=2)
    for attempt in range(10):
        try:
            os.replace(file.name, RAW_FILE)
            return
        except PermissionError:
            time.sleep(1)
    raise RuntimeError("Could not save JSON. Completed copy: " + file.name)

## Collect movies

In [24]:
page_number = 1
while len(raw["movie_details"]) < TARGET_MOVIES:
    page_key = str(page_number)
    if page_key in raw["pages"]:
        page = raw["pages"][page_key]
    else:
        page_url = "https://api.themoviedb.org/3/discover/movie"
        page_options = {"page": page_number, "language": "en-US"}
        page = get_json(page_url, page_options)
        raw["pages"][page_key] = page
        save_raw()
    films = page.get("results", [])
    if not films:
        break

    for film in films:
        movie_id = str(film["id"])
        if movie_id in raw["movie_details"]:
            continue

        movie_url = f"https://api.themoviedb.org/3/movie/{movie_id}"
        movie_options = {"language": "en-US", "append_to_response": "keywords"}
        movie_data = get_json(movie_url, movie_options)
        raw["movie_details"][movie_id] = movie_data
        save_raw()

        if len(raw["movie_details"]) >= TARGET_MOVIES:
            break

    print("Page", page_number, "- films saved:", len(raw["movie_details"]))
    total_pages = page["total_pages"]
    if page_number >= total_pages:
        break
    page_number += 1

print("Finished:", len(raw["movie_details"]), "films in", RAW_FILE)

Page 1 - films saved: 2956
Page 2 - films saved: 2956
Page 3 - films saved: 2956
Page 4 - films saved: 2956
Page 5 - films saved: 2956
Page 6 - films saved: 2956
Page 7 - films saved: 2956
Page 8 - films saved: 2956
Page 9 - films saved: 2956
Page 10 - films saved: 2956
Page 11 - films saved: 2956
Page 12 - films saved: 2956
Page 13 - films saved: 2956
Page 14 - films saved: 2956
Page 15 - films saved: 2956
Page 16 - films saved: 2956
Page 17 - films saved: 2956
Page 18 - films saved: 2956
Page 19 - films saved: 2956
Page 20 - films saved: 2956
Page 21 - films saved: 2956
Page 22 - films saved: 2956
Page 23 - films saved: 2956
Page 24 - films saved: 2956
Page 25 - films saved: 2956
Page 26 - films saved: 2956
Page 27 - films saved: 2956
Page 28 - films saved: 2956
Page 29 - films saved: 2956
Page 30 - films saved: 2956
Page 31 - films saved: 2956
Page 32 - films saved: 2956
Page 33 - films saved: 2956
Page 34 - films saved: 2956
Page 35 - films saved: 2956
Page 36 - films saved: 2956
P